# Reconstruction Channel V1 · Colab 测量入口

本 notebook 从公开仓库的固定源码版本读取 4 个固定来源：先收集首来源，工程收集完整后继续其余 3 个；首来源计入总数。响应为零或量化诊断无效仍继续，结果不用于筛选来源。只测一个固定 HF 家族探针与完整 VAE 特征，不编码、不拟合读者、不报告科学 PASS。

在 Colab 选择标准 GPU runtime，并将 `HF_TOKEN` 加入 Secrets，然后 Run all。源码和配置自动获取，不需要上传代码包。依赖组合仅经过静态/CPU 检查，尚未进行真实 GPU 验证。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. 获取固定源码与配置

固定配置只读显示。已有完整批次只展示结果；已有失败、未完成或配置不一致的批次会停止，不自动重试。源码保存在独立的 Colab 临时目录，测量结果保存在 Drive。


In [ ]:
import copy
import importlib
import json
import os
import pathlib
import re
import subprocess
import sys
import zipfile
from importlib.metadata import PackageNotFoundError, version

SOURCE_REPO = "https://github.com/RICHAAARC/CEG-WM.git"
CODE_REF = "85d6566a571c81205063d313398bf2e459473680"
CHECKOUT = pathlib.Path("/content/cegwm-reconstruction-channel-v1-published")
DRIVE_PARENT = pathlib.Path("/content/drive/MyDrive/CEG-WM/reconstruction-channel-v1")
RUN_ROOT = DRIVE_PARENT / "measurement-20261108"
CONFIG_RELATIVE = pathlib.Path("configs/reconstruction_channel_v1_colab.json")
REQUIREMENTS_RELATIVE = pathlib.Path("requirements-reconstruction-channel-colab.txt")
ARMS = ("zero", "plus", "minus")


def read_json(path):
    return json.loads(pathlib.Path(path).read_text(encoding="utf-8"))


def write_json(path, value):
    pathlib.Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + "\n", encoding="utf-8")


def fetch_source(repository, code_ref, checkout, run_root):
    checkout, run_root = pathlib.Path(checkout), pathlib.Path(run_root)
    if checkout.exists() and not (checkout / ".git").exists():
        raise RuntimeError("The dedicated source directory already contains non-repository data; it is retained: " + str(checkout))
    try:
        if not checkout.exists():
            subprocess.run(["git", "clone", "--no-checkout", repository, str(checkout)],
                           check=True, text=True, capture_output=True)
        subprocess.run(["git", "-C", str(checkout), "fetch", "origin", code_ref],
                       check=True, text=True, capture_output=True)
        subprocess.run(["git", "-C", str(checkout), "checkout", "--detach", "FETCH_HEAD"],
                       check=True, text=True, capture_output=True)
        resolved = subprocess.run(["git", "-C", str(checkout), "rev-parse", "HEAD"],
                                  check=True, text=True, capture_output=True).stdout.strip()
    except (OSError, subprocess.CalledProcessError) as error:
        detail = getattr(error, "stderr", None) or str(error)
        raise RuntimeError("Could not fetch the published source " + repository + " at " + code_ref
                           + ". Check Colab network access and that the published reference exists. "
                           + "Existing local files are retained. " + str(detail)) from error
    metadata = {
        "repository": repository, "requested_ref": code_ref, "resolved_commit": resolved,
        "scope": "Source checked out by this notebook; not proof of the producer of pre-existing batch results.",
        "preexisting_batch_directories": [name for name in ("first_source", "remaining_sources")
                                          if (run_root / name).exists()],
    }
    run_root.mkdir(parents=True, exist_ok=True)
    version_path = run_root / "code_version.json"
    if not version_path.exists():
        write_json(version_path, metadata)
    print({"source_checkout": str(checkout), "current_resolved_commit": resolved,
           "first_source_record": str(version_path)})
    return metadata


In [ ]:
SOURCE_VERSION = fetch_source(SOURCE_REPO, CODE_REF, CHECKOUT, RUN_ROOT)
CONFIG = read_json(CHECKOUT / CONFIG_RELATIVE)
if len(CONFIG.get("sources", [])) != 4:
    raise ValueError("This notebook requires the fixed four-source roster")
print(json.dumps(CONFIG, indent=2, ensure_ascii=False))
print({"source_ref": CODE_REF, "output": str(RUN_ROOT)})


## 2. 顺序收集固定四来源

下面调用固定源码中的 runner。每批只使用全新输出目录；已经完整的批次不再生成。仅在新批次前检查 GPU/依赖和读取 HF_TOKEN，不重装 Colab 的 CUDA PyTorch。首来源收集失败会停止后续来源；可直接运行最后的汇总格保留失败与缺失并下载结果。


In [ ]:
_runtime_env = None



def counts_are_terminal(summary):
    stages = summary.get("stages", {})
    if not stages or any(value == "started" for value in stages.values()):
        return False
    if any(row.get("status") == "started" or row.get("stage") in
           {"started", "generation", "generation_complete", "reconstruction"}
           for row in summary.get("sources", [])):
        return False
    if stages.get("cleanup") in {"complete", "failed", "caller_owned"}:
        return True
    # A caught loader failure can occur before a pipeline owner exists.
    return bool(summary.get("setup_error") and stages.get("generation_setup") == "failed")


def engineering_issues(output, expected_config):
    output = pathlib.Path(output)
    issues = []
    try:
        if read_json(output / "config.json") != expected_config:
            issues.append("saved config differs from the fixed batch config")
        summary = read_json(output / "summary.json")
    except (OSError, ValueError) as error:
        return ["missing or unreadable batch files: " + str(error)]
    expected_ids = [row["source_id"] for row in expected_config["sources"]]
    rows = summary.get("sources", [])
    if [row.get("source_id") for row in rows] != expected_ids:
        issues.append("source roster is incomplete or different")
    if summary.get("setup_error"):
        issues.append("runner setup failed")
    if not counts_are_terminal(summary) or summary.get("stages", {}).get("cleanup") not in {"complete", "caller_owned"}:
        issues.append("runner stages or cleanup are not successfully terminal")
    n, seeds = len(expected_ids), expected_config["reconstruction"]["seeds"]
    expected_counts = {
        "generation_attempted": 3 * n, "generation_returned": 3 * n,
        "reconstruction_attempted": 3 * n * len(seeds),
        "reconstruction_returned": 3 * n * len(seeds),
        "feature_reads_attempted": 3 * n * (1 + len(seeds)),
        "feature_reads_saved": 3 * n * (1 + len(seeds)),
    }
    if any(summary.get("actual_calls", {}).get(key) != value for key, value in expected_counts.items()):
        issues.append("recorded calls are not exactly the planned calls")
    for row in rows:
        if row.get("collection_complete") is not True or row.get("matched_state") is not True:
            issues.append(str(row.get("source_id")) + ": collection or replay state is incomplete")
        directory = output / str(row.get("source_id", ""))
        if not (directory / "measurement.json").is_file():
            issues.append(str(row.get("source_id")) + ": missing measurement record")
        for channel in ("clean", *(f"reconstruction_{seed}" for seed in seeds)):
            for arm in ARMS:
                for suffix in (".png", ".pt"):
                    if not (directory / channel / (arm + suffix)).is_file():
                        issues.append(str(row.get("source_id")) + ": missing " + channel + "/" + arm + suffix)
    return issues


def ensure_runtime(checkout, run_root):
    global _runtime_env
    if _runtime_env is not None:
        return dict(_runtime_env)
    torch = importlib.import_module("torch")
    parsed = re.match(r"(\d+)\.(\d+)", torch.__version__)
    runtime = {"torch": torch.__version__, "cuda_available": bool(torch.cuda.is_available())}
    pathlib.Path(run_root).mkdir(parents=True, exist_ok=True)
    if parsed is None or tuple(map(int, parsed.groups())) < (2, 6) or not runtime["cuda_available"]:
        write_json(pathlib.Path(run_root) / "environment.json", runtime)
        raise RuntimeError("Use a standard Colab GPU runtime with CUDA PyTorch >= 2.6; this notebook does not reinstall torch.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(pathlib.Path(checkout) / REQUIREMENTS_RELATIVE)], check=True)
    for package in ("diffusers", "transformers", "accelerate", "sentencepiece", "protobuf", "numpy", "Pillow"):
        try:
            runtime[package] = version(package)
        except PackageNotFoundError:
            runtime[package] = None
    write_json(pathlib.Path(run_root) / "environment.json", runtime)
    print(runtime)
    from google.colab import userdata
    token = os.environ.get("HF_TOKEN") or userdata.get("HF_TOKEN")
    if not token:
        raise RuntimeError("Add HF_TOKEN to Colab Secrets before starting a new batch.")
    environment = dict(os.environ)
    environment["PYTHONPATH"] = str(pathlib.Path(checkout) / "src") + os.pathsep + str(checkout)
    environment["HF_TOKEN"] = token
    _runtime_env = environment
    return dict(environment)


def record_launch_failure(output, config, message):
    # Protect a launch that failed before the runner could create its directory.
    # Existing runner evidence is never overwritten here.
    output = pathlib.Path(output)
    if not output.exists():
        output.mkdir(parents=True, exist_ok=False)
        write_json(output / "config.json", config)
        write_json(output / "summary.json", {
            "mode": "measurement", "setup_error": message, "actual_calls": {},
            "sources": [{"source_id": row["source_id"], "status": "launch_failed",
                         "collection_complete": False} for row in config["sources"]],
        })


def run_or_show_batch(name, batch_config, checkout, run_root):
    output = pathlib.Path(run_root) / name
    if output.exists():
        issues = engineering_issues(output, batch_config)
        if issues:
            raise RuntimeError("Existing batch is retained without retry: " + str(output) + "; " + "; ".join(issues))
        print({"batch": name, "action": "read_only_existing", "output": str(output)})
        return read_json(output / "summary.json")
    child_env = ensure_runtime(checkout, run_root)
    config_dir = pathlib.Path(checkout) / ".colab_batches"
    config_dir.mkdir(parents=True, exist_ok=True)
    config_path = config_dir / (name + ".json")
    if config_path.exists() and read_json(config_path) != batch_config:
        raise RuntimeError("Existing local batch config differs; no experiment was started.")
    if not config_path.exists():
        write_json(config_path, batch_config)
    command = [sys.executable, "-m", "experiments.run_reconstruction_channel_v1",
               "--config", str(config_path), "--execute", "--output", str(output)]
    try:
        completed = subprocess.run(command, cwd=checkout, env=child_env, check=False)
    except BaseException as error:
        record_launch_failure(output, batch_config, type(error).__name__ + ": " + str(error))
        raise
    if not output.exists():
        record_launch_failure(output, batch_config, "Runner exited before creating output; exit=" + str(completed.returncode))
    issues = engineering_issues(output, batch_config)
    if issues:
        raise RuntimeError("Batch retained; remaining sources will not start: " + "; ".join(issues))
    print({"batch": name, "collection_complete": True, "runner_exit": completed.returncode, "output": str(output)})
    return read_json(output / "summary.json")


def run_fixed_roster(config, checkout, run_root):
    run_root = pathlib.Path(run_root)
    run_root.mkdir(parents=True, exist_ok=True)
    full_config_path = run_root / "config.json"
    if full_config_path.exists() and read_json(full_config_path) != config:
        raise RuntimeError("Run-root config differs; no new batch will start.")
    if not full_config_path.exists():
        write_json(full_config_path, config)
    first, remaining = copy.deepcopy(config), copy.deepcopy(config)
    first["sources"], remaining["sources"] = config["sources"][:1], config["sources"][1:]
    if len(config["sources"]) != 4:
        raise ValueError("Expected exactly four fixed sources")
    first_summary = run_or_show_batch("first_source", first, checkout, run_root)
    remaining_summary = run_or_show_batch("remaining_sources", remaining, checkout, run_root)
    return first_summary, remaining_summary


In [ ]:
def aggregate_and_package(config, run_root):
    run_root = pathlib.Path(run_root)
    run_root.mkdir(parents=True, exist_ok=True)
    config_path = run_root / "config.json"
    if config_path.exists() and read_json(config_path) != config:
        raise RuntimeError("Run-root config differs; existing evidence is retained.")
    if not config_path.exists():
        write_json(config_path, config)
    batches, indexed, unknown_counts, last_recorded_counts = {}, {}, [], {}
    counts = dict.fromkeys(("generation_attempted", "generation_returned", "reconstruction_attempted",
                            "reconstruction_returned", "feature_reads_attempted", "feature_reads_saved"), 0)
    for name in ("first_source", "remaining_sources"):
        output = run_root / name
        if not output.exists():
            batches[name] = {"status": "not_attempted"}
            continue
        try:
            summary = read_json(output / "summary.json")
        except (OSError, ValueError) as error:
            summary = {"status": "unreadable", "error": str(error)}
        expected_config = copy.deepcopy(config)
        expected_config["sources"] = config["sources"][:1] if name == "first_source" else config["sources"][1:]
        try:
            saved_config = read_json(output / "config.json")
        except (OSError, ValueError):
            saved_config = None
        config_matches = saved_config == expected_config
        expected_ids = [row["source_id"] for row in expected_config["sources"]]
        roster_matches = [row.get("source_id") for row in summary.get("sources", [])] == expected_ids
        terminal = counts_are_terminal(summary)
        batches[name] = {"saved_config_matches": config_matches, "saved_config": saved_config,
                         "counts_terminal": terminal, "summary": summary}
        if config_matches and roster_matches:
            for row in summary.get("sources", []):
                indexed[row["source_id"]] = row
        else:
            reason = "config_mismatch" if not config_matches else "roster_mismatch"
            for identifier in expected_ids:
                indexed[identifier] = {"source_id": identifier, "status": reason, "collection_complete": False}
        last_recorded_counts[name] = summary.get("actual_calls", {})
        for key in counts:
            value = summary.get("actual_calls", {}).get(key)
            if config_matches and terminal and type(value) is int and value >= 0:
                if counts[key] is not None:
                    counts[key] += value
            else:
                counts[key] = None
                if name not in unknown_counts:
                    unknown_counts.append(name)
    n, seed_count = len(config["sources"]), len(config["reconstruction"]["seeds"])
    aggregate = {
        "scope": "fixed_four_source_measurement_only", "config": config,
        "planned_calls": {"generation_calls": 3 * n, "reconstruction_calls": 3 * n * seed_count,
                          "public_feature_calls": 3 * n * (1 + seed_count)},
        "actual_calls": counts, "unknown_actual_call_batches": unknown_counts,
        "last_recorded_counts": last_recorded_counts,
        "sources": [indexed.get(row["source_id"], {"source_id": row["source_id"], "status": "missing",
                                                  "collection_complete": False}) for row in config["sources"]],
        "batches": batches, "interpretation": "Collection and numerical diagnostics only; no scientific adjudication.",
    }
    write_json(run_root / "measurement_summary.json", aggregate)
    results_zip = run_root.parent / (run_root.name + "-results.zip")
    with zipfile.ZipFile(results_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(run_root.rglob("*")):
            if path.is_file() and not path.is_symlink() and path.resolve().is_relative_to(run_root.resolve()):
                archive.write(path, pathlib.Path(run_root.name) / path.relative_to(run_root))
    return aggregate, results_zip


In [ ]:
FIRST_SUMMARY, REMAINING_SUMMARY = run_fixed_roster(CONFIG, CHECKOUT, RUN_ROOT)


## 3. 汇总与回传

本格只读取已保存的批次，不运行模型。即使上一格报错，也可单独运行本格：固定四来源中未执行的来源仍列为缺失，失败不会被删除。下载生成的 ZIP 后回传本会话；完整配置、各臂图像/特征、调用记录和失败信息均保留。


In [ ]:
AGGREGATE, RESULTS_ZIP = aggregate_and_package(CONFIG, RUN_ROOT)
print(json.dumps({
    "planned_calls": AGGREGATE["planned_calls"],
    "actual_calls": AGGREGATE["actual_calls"],
    "sources": [{"source_id": row["source_id"], "status": row.get("status"),
                 "collection_complete": row.get("collection_complete", False)} for row in AGGREGATE["sources"]],
    "results_zip": str(RESULTS_ZIP),
}, indent=2, ensure_ascii=False))
from google.colab import files
files.download(str(RESULTS_ZIP))
